In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../Data/Raw/american_bankruptcy.csv")


df = df.rename(columns={
    "X1": "current_assets",
    "X2": "cost_of_goods_sold",
    "X3": "depreciation_amortization",
    "X4": "ebitda",
    "X5": "inventory",
    "X6": "net_income",
    "X7": "total_receivables",
    "X8": "market_value",
    "X9": "net_sales",
    "X10": "total_assets",
    "X11": "long_term_debt",
    "X12": "ebit",
    "X13": "gross_profit",
    "X14": "current_liabilities",
    "X15": "retained_earnings",
    "X16": "total_revenue",
    "X17": "total_liabilities",
    "X18": "operating_expenses"
})

df['bankrupt'] = df["status_label"].map({"alive": 0, "failed": 1})
df['bankrupt'] = df['bankrupt'].astype('int8')


In [10]:
fedfunds_df = pd.read_csv("../Data/Raw/FEDFUNDS.csv")
cpi_df = pd.read_csv("../Data/Raw/CPIAUCSL.csv")
unemployment_df = pd.read_csv("../Data/Raw/UNRATE.csv")
baa_df = pd.read_csv("../Data/Raw/BAA10Y.csv")

def clean_econ(df, value_col, value_name):
    df['observation_date'] = pd.to_datetime(df['observation_date'])
    df['year'] = df['observation_date'].dt.year
    df = df.groupby('year')[value_col].mean().reset_index()
    df = df.rename(columns={value_col: value_name})

    return df

fedfunds_df = clean_econ(fedfunds_df, 'FEDFUNDS', 'fed_funds_rate')
cpi_df = clean_econ(cpi_df, 'CPIAUCSL', 'cpi')
unemployment_df = clean_econ(unemployment_df, 'UNRATE', 'unemployment_rate')
baa_df = clean_econ(baa_df, 'BAA10Y', 'baa_rate')

macro_df = fedfunds_df.merge(cpi_df, on='year') \
    .merge(unemployment_df, on='year') \
    .merge(baa_df, on='year')


df['current_ratio'] = df['current_assets'] / df['current_liabilities']
df['debt_ratio'] = df['total_liabilities'] / df['total_assets']
df['net_profit_margin'] = df['net_income'] / df['total_revenue']
df['asset_turnover_ratio'] = df['total_revenue'] / df['total_assets']
df['inventory_turnover'] = df['cost_of_goods_sold'] / df['inventory']

df.replace([float("inf"), -float("inf")], 0, inplace=True)
df.fillna(0, inplace=True)

df = df.merge(macro_df, on='year', how='left')

df['interest_rate_pressure'] = df['debt_ratio'] * df['fed_funds_rate']

df.columns


Index(['company_name', 'status_label', 'year', 'current_assets',
       'cost_of_goods_sold', 'depreciation_amortization', 'ebitda',
       'inventory', 'net_income', 'total_receivables', 'market_value',
       'net_sales', 'total_assets', 'long_term_debt', 'ebit', 'gross_profit',
       'current_liabilities', 'retained_earnings', 'total_revenue',
       'total_liabilities', 'operating_expenses', 'bankrupt', 'current_ratio',
       'debt_ratio', 'net_profit_margin', 'asset_turnover_ratio',
       'inventory_turnover', 'fed_funds_rate', 'cpi', 'unemployment_rate',
       'baa_rate', 'interest_rate_pressure'],
      dtype='object')

In [11]:
df = df.drop(columns=['ebit', 'cost_of_goods_sold', 'current_assets', 'current_liabilities',
 'total_receivables','gross_profit', 'fed_funds_rate', 'depreciation_amortization',
 'net_sales', 'total_liabilities', 'total_assets', 'operating_expenses', 'inventory'])

df.columns

Index(['company_name', 'status_label', 'year', 'ebitda', 'net_income',
       'market_value', 'long_term_debt', 'retained_earnings', 'total_revenue',
       'bankrupt', 'current_ratio', 'debt_ratio', 'net_profit_margin',
       'asset_turnover_ratio', 'inventory_turnover', 'cpi',
       'unemployment_rate', 'baa_rate', 'interest_rate_pressure'],
      dtype='object')

In [12]:
df.to_csv("../Data/Cleaned/cleaned_data.csv", index = False)

In [13]:
print(df['bankrupt'])

0        0
1        0
2        0
3        0
4        0
        ..
78677    0
78678    0
78679    0
78680    0
78681    0
Name: bankrupt, Length: 78682, dtype: int8
